In [25]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Forcefully reload the new key from .env file
load_dotenv(override=True)

# Directly verify your key is loaded
print("Loaded Key starts with:", os.getenv("OPENAI_API_KEY")[:12])

model = init_chat_model("gpt-4o-mini", model_provider="openai")


Loaded Key starts with: sk-proj-n4eR


In [26]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [27]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.9', 'langchain': '1.4.4', 'langchain-openai': '1.7.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000224812F3ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000224813C87D0>, root_client=<openai.OpenAI objec

In [28]:
model.invoke("provide details about the movie inception")

AIMessage(content='"Inception" is a science fiction film written and directed by Christopher Nolan, released in 2010. The film blends elements of action, heist, and psychological drama, set against a backdrop of dreams and subconscious exploration.\n\n### Plot Summary\nThe story follows Dom Cobb (played by Leonardo DiCaprio), a skilled thief who specializes in extracting valuable secrets from within the subconscious during the dream state. Cobb is offered a chance to have his criminal history erased in exchange for performing "inception": the act of planting an idea in someone\'s mind without them realizing it. He is tasked with planting an idea into the mind of a business rival, Robert Fischer (Cillian Murphy), by creating a multi-layered dream environment.\n\nIn order to achieve this, Cobb assembles a team that includes:\n- **Arthur (Joseph Gordon-Levitt)**: Cobb\'s partner and point man.\n- **Ariadne (Elliot Page)**: A gifted architecture student responsible for designing the dream 

In [29]:
response=model_with_structure.invoke("Provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output alognside parsed structure

In [30]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response=model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 117, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8794d3bcd1', 'id': 'chatcmpl-EZFeWQ9AWhI9NM8dRTMrYxEPkbzxX', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13fb7-85fa-7b71-830c-40a7c85ee0ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_toke

### Nested Structure

In [32]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budge: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response   

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Sci-Fi', 'Thriller'], budge=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, iideal when you don't need runtime validation.

In [33]:
from typing_extensions import TypedDict, Annotated

class MovideDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "the year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypeddict = model.with_structured_output(MovideDict)
response = model_withtypeddict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [34]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response   

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Marion Cotillard', 'role': 'Mal Cobb'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Tom Berenger', 'role': 'Browning'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'budget': 160000000}

In [36]:
model.profile

{'name': 'GPT-4o mini',
 'release_date': '2024-07-18',
 'last_updated': '2024-07-18',
 'open_weights': False,
 'max_input_tokens': 128000,
 'max_output_tokens': 16384,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions.You create it using the @dataclass decorator 

In [38]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Contantinfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description = "The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=Contantinfo
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9b88b318-fde9-4234-b46f-517b74915471'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1061, 'prompt_tokens': 204, 'total_tokens': 1265, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1024, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZFyZgezHW9kdxPiVMDKZZq8FKO1U', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13fca-82d4-77a2-8d12-863b6e434cd5

In [39]:
result["structured_response"]

Contantinfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [41]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class Contantinfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description = "The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=Contantinfo
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='04ef97fc-2afa-4b43-8e1f-b53e5634d4b1'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 869, 'prompt_tokens': 178, 'total_tokens': 1047, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZG3ue9WtoxO0d4shyOGE8nVK5zBp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13fcf-8ec9-7500-b81a-faac2338cc53-0

In [42]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contantinfo:
    """Contact information for a person"""
    name: str 
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=Contantinfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

Contantinfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')